# Decision Trees — Implementations

One extra lane. There is no torch lane anywhere in this topic — a tree is fit by a discrete argmax over candidate thresholds, and no gradient flows through “which side of the split a point falls on”, so autograd has nothing to act on. The honest comparison is sklearn's CART, and because two correct tree learners may tie-break ambiguous splits differently, the equivalence fixture uses well-separated blobs on which every reasonable tree predicts identically, compared on integer predictions over a fixed probe grid plus train accuracy.

## 05_decision_tree_classifier

Greedy recursive splitting on impurity. **No torch lane:** the split search is a discrete argmax over features and thresholds — a piecewise-constant objective with zero gradient almost everywhere — so there is nothing for autograd to differentiate.

### library

sklearn's `DecisionTreeClassifier` with the notebook's parameters: `criterion='gini'`, the same `max_depth`, the same midpoint thresholds. **What the library adds:** an optimized Cython splitter plus pruning, class weights and feature importances — the algorithm itself is the one the notebook already built, as the checks on impurity and depth confirm.

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

# hints:
# 1. DecisionTreeClassifier(criterion='gini', max_depth=...) mirrors the scratch tree.
# 2. sklearn sends x <= threshold left, the scratch tree x < — midpoints make it moot.
# 3. sklearn casts X to float32 inside, so thresholds match midpoints only to ~1e-7.
# 4. random_state only breaks ties between equally good splits; clean gaps have none.
# 5. The fitted tree is inspectable: tree_.feature, tree_.threshold, tree_.impurity.


class ScratchDecisionTreeClassifier:
    """sklearn's CART behind the notebook's interface.

    Same greedy recipe: for every feature, sort, scan the midpoints between
    consecutive distinct values, keep the split with the largest impurity
    decrease, recurse. On data with clean gaps the two implementations build
    the same tree; on ambiguous data they may tie-break differently, which is
    why the equivalence fixture uses well-separated blobs."""

    def __init__(self, max_depth=None, min_samples_split=2, criterion="gini"):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.criterion = criterion

    def fit(self, X, y):
        self._model = DecisionTreeClassifier(
            criterion=self.criterion, max_depth=self.max_depth,
            min_samples_split=self.min_samples_split, random_state=0)
        self._model.fit(np.asarray(X, dtype=float), np.asarray(y))
        self.classes_ = self._model.classes_
        return self

    def predict(self, X):
        return self._model.predict(np.asarray(X, dtype=float))

    def score(self, X, y):
        return float(np.mean(self.predict(X) == np.asarray(y)))


In [ ]:
# exports: probe_pred, train_acc
_rng_eq = np.random.default_rng(505)
_centers_eq = np.repeat(np.array([[-3.0, 0.0], [3.0, -3.0], [3.0, 3.0]]), [25, 20, 15], axis=0)
X_tree_eq = _centers_eq + _rng_eq.uniform(-0.8, 0.8, size=(60, 2))
y_tree_eq = np.repeat(np.array([0, 1, 2]), [25, 20, 15])
_ticks_eq = np.array([-3.5, -3.0, -2.5, 2.5, 3.0, 3.5])
_gx_eq, _gy_eq = np.meshgrid(_ticks_eq, _ticks_eq)
X_probe_eq = np.column_stack([_gx_eq.ravel(), _gy_eq.ravel()])

_fit_eq = ScratchDecisionTreeClassifier(max_depth=3, criterion="gini").fit(X_tree_eq, y_tree_eq)
probe_pred = _fit_eq.predict(X_probe_eq).astype(int)
train_acc = _fit_eq.score(X_tree_eq, y_tree_eq)
print("train acc:", train_acc, "| depth:", _fit_eq._model.get_depth(),
      "| leaves:", _fit_eq._model.get_n_leaves())


In [ ]:
assert train_acc == 1.0, "three separated blobs are classified perfectly"
assert _fit_eq._model.get_depth() == 2, "two greedy splits are enough for three blobs"

# The impurity sklearn reports at the root is the notebook's gini, verbatim.
_p_root = np.bincount(y_tree_eq) / len(y_tree_eq)
assert abs(_fit_eq._model.tree_.impurity[0] - (1.0 - np.sum(_p_root ** 2))) < 1e-12

# Trees compare, never measure: a monotone map of the features changes nothing.
_fit_mono = ScratchDecisionTreeClassifier(max_depth=3).fit(3.0 * X_tree_eq + 1.0, y_tree_eq)
assert np.array_equal(_fit_mono.predict(3.0 * X_probe_eq + 1.0), probe_pred), \
    "predictions are invariant under monotone feature transforms"

# A depth-1 stump gets one split: it peels off class 0 and never predicts class 2.
_stump = ScratchDecisionTreeClassifier(max_depth=1).fit(X_tree_eq, y_tree_eq)
assert _stump.score(X_tree_eq, y_tree_eq) == 0.75 and set(_stump.predict(X_tree_eq)) == {0, 1}
